# Milestone 1: Tabular MLP

This notebook predicts a product rating band from tabular metadata and lightweight review-derived features.

**Inputs:**
- `data/processed/train.csv`
- `data/processed/validation.csv`
- `data/processed/test.csv`

**Models:**
- Baseline: Logistic Regression
- Neural model: Keras MLP

Milestone 2 is intentionally not implemented here.

## 1. Setup

The notebook is self-contained and can run top-to-bottom after Milestone 0 has generated the processed CSV files.

In [ ]:
from __future__ import annotations

import re
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedKFold, learning_curve
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
sns.set_theme(style='whitegrid')
tf.keras.utils.set_random_seed(42)

RANDOM_SEED = 42
LABEL_ORDER = ['low', 'medium', 'high']
FEATURE_COLUMNS = [
    'price',
    'rating_count',
    'mean_review_length',
    'has_image',
    'helpful_vote_mean',
    'helpful_vote_total',
]

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
SPLIT_PATHS = {
    'train': PROCESSED_DIR / 'train.csv',
    'validation': PROCESSED_DIR / 'validation.csv',
    'test': PROCESSED_DIR / 'test.csv',
}

print(f'Project root: {PROJECT_ROOT}')
print(f'Processed data directory: {PROCESSED_DIR}')

## 2. Load Processed Splits

Milestone 1 starts from the leakage-safe product split created in Milestone 0. If any file is missing, rerun `notebooks/00_eda.ipynb` first.

In [ ]:
missing_files = [path for path in SPLIT_PATHS.values() if not path.exists()]
if missing_files:
    missing_text = '\n'.join(str(path) for path in missing_files)
    raise FileNotFoundError(f'Missing processed split files. Run notebooks/00_eda.ipynb first:\n{missing_text}')

splits = {name: pd.read_csv(path, low_memory=False) for name, path in SPLIT_PATHS.items()}

for name, df in splits.items():
    print(f'{name}: {df.shape[0]:,} rows, {df.shape[1]:,} columns')
    display(pd.DataFrame({'column': df.columns.tolist()}))
    display(df.head(3))

## 3. Product-Level Feature Engineering

The processed CSVs are review-level rows. For a product rating-band task, we aggregate each split to one row per `product_id`. This preserves the Milestone 0 product split and avoids mixing the same product across train, validation, and test.

Target construction prefers product `average_rating`. When product metadata is unavailable, it falls back to the product mean of `review_rating` so the notebook can still run on sparse metadata joins.

In [ ]:
def first_existing_column(df: pd.DataFrame, candidates: list[str]) -> str | None:
    return next((column for column in candidates if column in df.columns), None)


def extract_price(value) -> float:
    if value is None:
        return np.nan
    if isinstance(value, float) and np.isnan(value):
        return np.nan
    if isinstance(value, (int, float, np.integer, np.floating)):
        return float(value)
    text = str(value).replace(',', '').strip()
    match = re.search(r'[0-9]+(?:[.][0-9]+)?', text)
    return float(match.group(0)) if match else np.nan


def numeric_series(df: pd.DataFrame, candidates: list[str], default_value=np.nan) -> pd.Series:
    column = first_existing_column(df, candidates)
    if column is None:
        return pd.Series([default_value] * len(df), index=df.index, dtype='float64')
    if column in {'price', 'price_raw', 'price_numeric'}:
        return df[column].map(extract_price).astype('float64')
    return pd.to_numeric(df[column], errors='coerce')


def bool_value(value) -> bool:
    if isinstance(value, bool):
        return value
    if value is None:
        return False
    if isinstance(value, float) and np.isnan(value):
        return False
    if isinstance(value, (int, np.integer)):
        return bool(value)
    return str(value).strip().lower() in {'true', '1', 'yes', 'y'}


def bool_series(df: pd.DataFrame, candidates: list[str]) -> pd.Series:
    column = first_existing_column(df, candidates)
    if column is None:
        return pd.Series([False] * len(df), index=df.index)
    return df[column].map(bool_value)


def make_rating_band(rating: pd.Series) -> pd.Series:
    clipped = pd.to_numeric(rating, errors='coerce').clip(lower=1, upper=5)
    return pd.cut(
        clipped,
        bins=[0, 3, 4, 5.01],
        labels=LABEL_ORDER,
        right=False,
        include_lowest=True,
    ).astype('string')


def build_product_table(df: pd.DataFrame, split_name: str) -> pd.DataFrame:
    if 'product_id' not in df.columns:
        raise ValueError(f'{split_name} is missing product_id. Rerun Milestone 0.')

    work = pd.DataFrame(index=df.index)
    work['product_id'] = df['product_id'].astype('string').str.strip()
    work['price'] = numeric_series(df, ['price_numeric', 'price', 'price_raw'])
    work['rating_count'] = numeric_series(df, ['rating_number', 'rating_count', 'review_count'])
    work['review_length'] = numeric_series(df, ['review_length'])
    work['has_product_image'] = bool_series(df, ['has_product_image', 'product_has_image'])
    work['has_review_image'] = bool_series(df, ['has_review_image', 'review_has_image'])
    work['has_image'] = work['has_product_image'] | work['has_review_image']
    work['helpful_vote'] = numeric_series(df, ['helpful_vote', 'helpful_votes'])
    work['average_rating'] = numeric_series(df, ['average_rating', 'product_average_rating'])
    work['review_rating'] = numeric_series(df, ['review_rating', 'rating', 'overall'])
    work = work.dropna(subset=['product_id'])
    work = work[work['product_id'].str.len() > 0]

    product_table = work.groupby('product_id', as_index=False).agg(
        price=('price', 'median'),
        rating_count=('rating_count', 'median'),
        mean_review_length=('review_length', 'mean'),
        has_image=('has_image', 'max'),
        helpful_vote_mean=('helpful_vote', 'mean'),
        helpful_vote_total=('helpful_vote', lambda values: values.sum(min_count=1)),
        product_average_rating=('average_rating', 'median'),
        mean_review_rating=('review_rating', 'mean'),
        review_rows=('review_rating', 'size'),
    )
    product_table['split'] = split_name
    product_table['target_rating'] = product_table['product_average_rating'].where(
        product_table['product_average_rating'].notna(),
        product_table['mean_review_rating'],
    )
    product_table['target_source'] = np.where(
        product_table['product_average_rating'].notna(),
        'average_rating',
        'mean_review_rating',
    )
    product_table.loc[product_table['target_rating'].isna(), 'target_source'] = 'missing'
    product_table['rating_band'] = make_rating_band(product_table['target_rating'])
    product_table['has_image'] = product_table['has_image'].astype(float)
    return product_table


product_splits = {name: build_product_table(df, name) for name, df in splits.items()}

summary = pd.DataFrame(
    [
        {
            'split': name,
            'products': len(df),
            'products_with_target': df['rating_band'].notna().sum(),
            'average_rating_targets': (df['target_source'] == 'average_rating').sum(),
            'review_mean_targets': (df['target_source'] == 'mean_review_rating').sum(),
        }
        for name, df in product_splits.items()
    ]
)
display(summary)
display(product_splits['train'].head())

## 4. Preprocessing

All model features are numeric after aggregation. Missing values are imputed from the training split, then features are standardized. Empty numeric metadata columns are kept and imputed so the notebook does not fail when metadata coverage is sparse.

In [ ]:
def prepare_model_frame(df: pd.DataFrame, known_labels: set[str] | None = None) -> pd.DataFrame:
    model_df = df.dropna(subset=['rating_band']).copy()
    model_df['rating_band'] = model_df['rating_band'].astype(str)
    if known_labels is not None:
        before = len(model_df)
        model_df = model_df[model_df['rating_band'].isin(known_labels)].copy()
        dropped = before - len(model_df)
        if dropped:
            print(f'Dropped {dropped:,} rows with labels not seen in training.')
    return model_df


train_model_df = prepare_model_frame(product_splits['train'])
observed_labels = [label for label in LABEL_ORDER if label in set(train_model_df['rating_band'])]

if len(observed_labels) < 2:
    raise ValueError(
        'Training data must contain at least two rating bands. Check Milestone 0 splits or rating-band thresholds.'
    )

known_labels = set(observed_labels)
validation_model_df = prepare_model_frame(product_splits['validation'], known_labels)
test_model_df = prepare_model_frame(product_splits['test'], known_labels)

if validation_model_df.empty or test_model_df.empty:
    raise ValueError('Validation and test splits must contain labels observed in training.')

label_to_id = {label: idx for idx, label in enumerate(observed_labels)}
id_to_label = {idx: label for label, idx in label_to_id.items()}


def make_xy(df: pd.DataFrame) -> tuple[pd.DataFrame, np.ndarray]:
    X = df[FEATURE_COLUMNS].copy()
    for column in FEATURE_COLUMNS:
        X[column] = pd.to_numeric(X[column], errors='coerce')
    X = X.replace([np.inf, -np.inf], np.nan)
    y = df['rating_band'].map(label_to_id).astype(int).to_numpy()
    return X, y


X_train, y_train = make_xy(train_model_df)
X_validation, y_validation = make_xy(validation_model_df)
X_test, y_test = make_xy(test_model_df)

print('Feature columns:', FEATURE_COLUMNS)
print('Labels:', observed_labels)
display(train_model_df['rating_band'].value_counts().reindex(observed_labels))
display(X_train.describe().T)

## 5. Baseline Model: Logistic Regression

The baseline uses the same preprocessing pattern as the neural model: median imputation and feature scaling.

In [ ]:
def make_preprocessor() -> Pipeline:
    return Pipeline(
        steps=[
            ('imputer', SimpleImputer(strategy='median', keep_empty_features=True)),
            ('scaler', StandardScaler()),
        ]
    )


baseline_model = Pipeline(
    steps=[
        ('preprocess', make_preprocessor()),
        (
            'model',
            LogisticRegression(
                max_iter=1000,
                class_weight='balanced',
                random_state=RANDOM_SEED,
            ),
        ),
    ]
)

baseline_model.fit(X_train, y_train)
baseline_validation_pred = baseline_model.predict(X_validation)
baseline_test_pred = baseline_model.predict(X_test)

print('Baseline Logistic Regression fitted.')

### Baseline Learning Curve

This curve estimates macro-F1 behavior as the logistic regression baseline sees larger portions of the training data.

In [ ]:
def plot_baseline_learning_curve(model: Pipeline, X: pd.DataFrame, y: np.ndarray) -> None:
    class_counts = pd.Series(y).value_counts()
    min_class_count = int(class_counts.min())
    if min_class_count < 2:
        print('Skipping baseline learning curve because at least one class has fewer than 2 samples.')
        return

    n_splits = min(3, min_class_count)
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED)
    train_sizes, train_scores, validation_scores = learning_curve(
        model,
        X,
        y,
        train_sizes=np.linspace(0.2, 1.0, 5),
        cv=cv,
        scoring='f1_macro',
        n_jobs=-1,
    )

    plt.figure(figsize=(8, 5))
    plt.plot(train_sizes, train_scores.mean(axis=1), marker='o', label='Train macro-F1')
    plt.plot(train_sizes, validation_scores.mean(axis=1), marker='o', label='CV macro-F1')
    plt.title('Logistic Regression Learning Curve')
    plt.xlabel('Training examples')
    plt.ylabel('Macro-F1')
    plt.legend()
    plt.tight_layout()
    plt.show()


plot_baseline_learning_curve(baseline_model, X_train, y_train)

## 6. Keras MLP Model

The MLP receives the same imputed and scaled tabular features. It uses ReLU hidden layers, dropout, and a softmax output layer for rating-band classification.

In [ ]:
mlp_preprocessor = make_preprocessor()
X_train_scaled = mlp_preprocessor.fit_transform(X_train)
X_validation_scaled = mlp_preprocessor.transform(X_validation)
X_test_scaled = mlp_preprocessor.transform(X_test)


def build_mlp(input_dim: int, num_classes: int) -> tf.keras.Model:
    inputs = tf.keras.Input(shape=(input_dim,), name='tabular_features')
    x = tf.keras.layers.Dense(64, activation='relu', name='dense_64')(inputs)
    x = tf.keras.layers.Dropout(0.30, name='dropout_1')(x)
    x = tf.keras.layers.Dense(32, activation='relu', name='dense_32')(x)
    x = tf.keras.layers.Dropout(0.20, name='dropout_2')(x)
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax', name='rating_band')(x)
    model = tf.keras.Model(inputs=inputs, outputs=outputs, name='tabular_rating_band_mlp')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )
    return model


mlp_model = build_mlp(input_dim=X_train_scaled.shape[1], num_classes=len(observed_labels))
mlp_model.summary()

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=8,
    restore_best_weights=True,
)

batch_size = min(64, max(8, len(X_train_scaled) // 20))
history = mlp_model.fit(
    X_train_scaled,
    y_train,
    validation_data=(X_validation_scaled, y_validation),
    epochs=50,
    batch_size=batch_size,
    callbacks=[early_stopping],
    verbose=1,
)

mlp_validation_proba = mlp_model.predict(X_validation_scaled, verbose=0)
mlp_test_proba = mlp_model.predict(X_test_scaled, verbose=0)
mlp_validation_pred = mlp_validation_proba.argmax(axis=1)
mlp_test_pred = mlp_test_proba.argmax(axis=1)

### MLP Learning Curves

The training history shows how loss and accuracy evolve across epochs.

In [ ]:
history_df = pd.DataFrame(history.history)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history_df['loss'], label='Train loss')
axes[0].plot(history_df['val_loss'], label='Validation loss')
axes[0].set_title('MLP Loss Curve')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()

axes[1].plot(history_df['accuracy'], label='Train accuracy')
axes[1].plot(history_df['val_accuracy'], label='Validation accuracy')
axes[1].set_title('MLP Accuracy Curve')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()

plt.tight_layout()
plt.show()

## 7. Model Comparison

We compare validation and test accuracy, macro-F1, and weighted-F1 for the logistic regression baseline and the MLP.

In [ ]:
def collect_metrics(model_name: str, split_name: str, y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float | str]:
    return {
        'model': model_name,
        'split': split_name,
        'accuracy': accuracy_score(y_true, y_pred),
        'macro_f1': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'weighted_f1': f1_score(y_true, y_pred, average='weighted', zero_division=0),
    }


comparison = pd.DataFrame(
    [
        collect_metrics('Logistic Regression', 'validation', y_validation, baseline_validation_pred),
        collect_metrics('Logistic Regression', 'test', y_test, baseline_test_pred),
        collect_metrics('Keras MLP', 'validation', y_validation, mlp_validation_pred),
        collect_metrics('Keras MLP', 'test', y_test, mlp_test_pred),
    ]
)
display(comparison)

## 8. Confusion Matrix and Classification Report

The confusion matrices show where each model confuses low, medium, and high product rating bands.

In [ ]:
def plot_confusion(ax, y_true: np.ndarray, y_pred: np.ndarray, title: str) -> None:
    labels = list(range(len(observed_labels)))
    matrix = confusion_matrix(y_true, y_pred, labels=labels)
    sns.heatmap(
        matrix,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=observed_labels,
        yticklabels=observed_labels,
        ax=ax,
    )
    ax.set_title(title)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')


fig, axes = plt.subplots(1, 2, figsize=(13, 5))
plot_confusion(axes[0], y_test, baseline_test_pred, 'Logistic Regression Test Confusion Matrix')
plot_confusion(axes[1], y_test, mlp_test_pred, 'Keras MLP Test Confusion Matrix')
plt.tight_layout()
plt.show()


def report_frame(y_true: np.ndarray, y_pred: np.ndarray) -> pd.DataFrame:
    report = classification_report(
        y_true,
        y_pred,
        labels=list(range(len(observed_labels))),
        target_names=observed_labels,
        zero_division=0,
        output_dict=True,
    )
    return pd.DataFrame(report).T


print('Logistic Regression classification report:')
display(report_frame(y_test, baseline_test_pred))

print('Keras MLP classification report:')
display(report_frame(y_test, mlp_test_pred))

## 9. Error Analysis

This section inspects products where predictions fail, including high-confidence MLP errors and disagreement between the baseline and neural model.

In [ ]:
def decode_labels(encoded: np.ndarray) -> list[str]:
    return [id_to_label[int(value)] for value in encoded]


analysis_columns = ['product_id', 'target_rating', 'target_source', 'rating_band', 'review_rows'] + FEATURE_COLUMNS
error_analysis = test_model_df[analysis_columns].reset_index(drop=True).copy()
error_analysis['baseline_pred'] = decode_labels(baseline_test_pred)
error_analysis['mlp_pred'] = decode_labels(mlp_test_pred)
error_analysis['mlp_confidence'] = mlp_test_proba.max(axis=1)
error_analysis['baseline_correct'] = error_analysis['baseline_pred'] == error_analysis['rating_band']
error_analysis['mlp_correct'] = error_analysis['mlp_pred'] == error_analysis['rating_band']
error_analysis['models_disagree'] = error_analysis['baseline_pred'] != error_analysis['mlp_pred']

print('Error summary:')
display(
    pd.DataFrame(
        {
            'metric': ['baseline_errors', 'mlp_errors', 'model_disagreements'],
            'count': [
                (~error_analysis['baseline_correct']).sum(),
                (~error_analysis['mlp_correct']).sum(),
                error_analysis['models_disagree'].sum(),
            ],
        }
    )
)

print('Highest-confidence MLP errors:')
display(
    error_analysis[~error_analysis['mlp_correct']]
    .sort_values('mlp_confidence', ascending=False)
    .head(20)
)

print('Baseline vs MLP prediction crosstab:')
display(pd.crosstab(error_analysis['baseline_pred'], error_analysis['mlp_pred'], rownames=['baseline'], colnames=['mlp']))

## Milestone 1 Complete

This notebook completes the first modeling milestone:

- Built a product-level tabular dataset from the Milestone 0 splits.
- Handled missing values and scaled numeric features.
- Trained a Logistic Regression baseline.
- Trained a Keras MLP with ReLU hidden layers and dropout.
- Compared metrics, learning curves, confusion matrices, classification reports, and error cases.

Milestone 2 is not implemented here.